# Gujarati Phase 3 — SFT LATEST / FIXED

This notebook fine-tunes the Gujarati Phase-2 GPT-style decoder-only Transformer on the Phase-3 synthetic reasoning dataset.

**Critical fixes:**
- Exact Gujarati Phase-2 architecture is written and force-reloaded: 10K vocab, 7 layers, 512 hidden, 8 heads, FFN 1792.
- Exact Phase-2 parameter count is checked: 25,613,056.
- Fresh Phase-3 AdamW optimizer; Phase-2 optimizer state is NOT resumed.
- EOS (`</s>`, ID 2) is explicitly appended to every supervised answer.
- Answer-only loss includes the final EOS token.
- Existing Phase-2 model API is respected: `model(input_ids)` returns a tuple and `outputs[0]` is used as logits. No `labels=` argument is passed to the model.
- Modern `torch.amp` API is used.
- Ordering evaluation keeps the complete comma-separated answer instead of truncating to the first entity.
- Raw ordering generation is checked before full test evaluation.
- The original Phase-2 checkpoint is never overwritten.


In [106]:
import os
import json
import math
import time
import random
from pathlib import Path
from collections import defaultdict

import torch
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Device: cuda


In [107]:
# ============================================================
# PATH DISCOVERY
# ============================================================

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

def find_file(filename, roots=SEARCH_ROOTS):
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        return None
    # Prefer Kaggle input over working copies when possible.
    matches = sorted(matches, key=lambda p: (0 if "/kaggle/input" in str(p) else 1, len(str(p))))
    return matches[0]

checkpoint_path = find_file("checkpoint_030518_gujarati.pt")
tokenizer_path = find_file("gujarati_10k.model")
train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
test_path = find_file("test.jsonl")

print("Checkpoint:", checkpoint_path)
print("Tokenizer :", tokenizer_path)
print("Train     :", train_path)
print("Val       :", val_path)
print("Test      :", test_path)

assert checkpoint_path is not None, "Phase-2 Gujarati checkpoint not found."
assert tokenizer_path is not None, "Gujarati tokenizer not found."
assert train_path is not None, "Phase-3 train.jsonl not found."
assert val_path is not None, "Phase-3 val.jsonl not found."
assert test_path is not None, "Phase-3 test.jsonl not found."

Checkpoint: /kaggle/input/datasets/vidurshah04/guj-ckcpts-and-json/checkpoint_030518_gujarati.pt
Tokenizer : /kaggle/input/datasets/vidurshah04/py-files/gujarati_10k.model
Train     : /kaggle/input/datasets/vidurshah04/guj-ckcpts-and-json/train.jsonl
Val       : /kaggle/input/datasets/vidurshah04/guj-ckcpts-and-json/val.jsonl
Test      : /kaggle/input/datasets/vidurshah04/guj-ckcpts-and-json/test.jsonl


## Kaggle input requirement

You need only:

- `checkpoint_030518_gujarati.pt`
- `gujarati_10k.model`
- `train.jsonl`
- `val.jsonl`
- `test.jsonl`
- `config.py`
- `transformer.py`

The large **`guj lma` Phase-2 pretraining corpus is NOT needed** for SFT.

In [108]:
# ============================================================
# LOAD EXACT PHASE-2 GUJARATI MODEL CODE
# ============================================================

import sys
import shutil

MODEL_CODE_DIR = Path("/kaggle/working/phase2_model_code")
MODEL_CODE_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: use the Phase-2 transformer.py, but write the exact
# Gujarati Phase-2 config used by checkpoint_030518_gujarati.pt.
transformer_path = find_file("transformer.py")
assert transformer_path is not None, "transformer.py not found."

config_text = """
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size: int = 10_000
    d_model: int = 512
    n_layers: int = 7
    n_heads: int = 8
    d_ff: int = 1792
    context_length: int = 512
    dropout: float = 0.1

    @property
    def head_dim(self):
        return self.d_model // self.n_heads
"""

(MODEL_CODE_DIR / "config.py").write_text(config_text)
shutil.copy2(transformer_path, MODEL_CODE_DIR / "transformer.py")

# Force fresh imports so an old 24K/5-layer config cannot remain cached.
sys.path.insert(0, str(MODEL_CODE_DIR))
sys.modules.pop("config", None)
sys.modules.pop("transformer", None)

from config import ModelConfig
from transformer import DecoderOnlyTransformer

print("Config: exact Gujarati Phase-2 architecture")
print("Transformer source:", transformer_path)


Config: exact Gujarati Phase-2 architecture
Transformer source: /kaggle/input/datasets/vidurshah04/py-files/transformer.py


In [109]:
# ============================================================
# MODEL CONFIG + ARCHITECTURE CHECK
# ============================================================

config = ModelConfig()
print(config)

model = DecoderOnlyTransformer(config)
num_params = sum(p.numel() for p in model.parameters())

print(f"Parameter count: {num_params:,}")

assert config.vocab_size == 10_000
assert config.d_model == 512
assert config.n_layers == 7
assert config.n_heads == 8
assert config.d_ff == 1792
assert config.context_length == 512
assert num_params == 25_613_056, (
    f"Unexpected Phase-2 architecture: {num_params:,}"
)

print("Exact Phase-2 architecture check: PASS")


ModelConfig(vocab_size=10000, d_model=512, n_layers=7, n_heads=8, d_ff=1792, context_length=512, dropout=0.1)
Parameter count: 25,613,056
Exact Phase-2 architecture check: PASS


In [110]:
# ============================================================
# LOAD PHASE-2 WEIGHTS ONLY
# ============================================================

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

print("Checkpoint keys:", checkpoint.keys())

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Phase-2 model weights loaded successfully.")

model = model.to(device)

Checkpoint keys: dict_keys(['step', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'config', 'loss', 'val_loss', 'learning_rate', 'dataset_state'])
Phase-2 model weights loaded successfully.


In [111]:
# ============================================================
# TOKENIZER
# ============================================================

import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load(str(tokenizer_path))

print("Vocab size:", sp.vocab_size())
print("EOS ID:", sp.eos_id())
print("EOS piece:", sp.id_to_piece(sp.eos_id()))
print("PAD ID:", sp.pad_id())

assert sp.vocab_size() == 10_000
assert sp.eos_id() >= 0

Vocab size: 10000
EOS ID: 2
EOS piece: </s>
PAD ID: 3


In [112]:
# ============================================================
# LOAD JSONL
# ============================================================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(train_path)
val_rows = load_jsonl(val_path)
test_rows = load_jsonl(test_path)

print("Train:", len(train_rows))
print("Val  :", len(val_rows))
print("Test :", len(test_rows))

assert len(train_rows) == 6400
assert len(val_rows) == 800
assert len(test_rows) == 800

Train: 6400
Val  : 800
Test : 800


In [113]:
# ============================================================
# SFT DATASET — EOS FIX
# ============================================================

SEQ_LEN = 512

class SFTDataset(Dataset):

    def __init__(self, rows, tokenizer, seq_len=512):
        self.rows = rows
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.eos_id = tokenizer.eos_id()
        self.pad_id = tokenizer.pad_id()

        if self.eos_id < 0:
            raise RuntimeError("Tokenizer has no EOS token.")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        prompt = row["input"]
        answer = row["answer"]

        # Prompt
        prompt_text = prompt + "\nજવાબ: "
        prompt_ids = self.tokenizer.encode(
            prompt_text,
            out_type=int
        )

        # Answer
        answer_ids = self.tokenizer.encode(
            answer,
            out_type=int
        )

        # ====================================================
        # CRITICAL FIX:
        # Explicitly teach the model to terminate after answer.
        # ====================================================
        answer_ids.append(self.eos_id)

        # Keep sequence within context length.
        answer_ids = answer_ids[:self.seq_len - 1]

        max_prompt_len = self.seq_len - len(answer_ids)
        prompt_ids = prompt_ids[:max_prompt_len]

        full_ids = prompt_ids + answer_ids

        # Causal LM shift
        input_ids = full_ids[:-1]
        labels = full_ids[1:]

        # Answer-only loss.
        # The first answer token is predicted at the last prompt position.
        prompt_target_len = max(len(prompt_ids) - 1, 0)

        labels = (
            [-100] * prompt_target_len
            + answer_ids
        )

        labels = labels[:len(input_ids)]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

In [114]:
# ============================================================
# DATASETS
# ============================================================

train_dataset = SFTDataset(train_rows, sp, SEQ_LEN)
val_dataset = SFTDataset(val_rows, sp, SEQ_LEN)
test_dataset = SFTDataset(test_rows, sp, SEQ_LEN)

print("Datasets created.")

Datasets created.


In [115]:
# ============================================================
# MANDATORY EOS SANITY CHECK
# ============================================================

row = train_rows[0]
sample = train_dataset[0]

target_ids = sample["labels"][
    sample["labels"] != -100
].tolist()

print("INPUT:")
print(row["input"])

print("\nGOLD ANSWER:")
print(row["answer"])

print("\nTARGET IDS:")
print(target_ids)

print("\nTARGET DECODED:")
print(sp.decode(target_ids))

print("\nEOS ID:", sp.eos_id())
print("LAST TARGET ID:", target_ids[-1])
print("LAST TARGET IS EOS?:", target_ids[-1] == sp.eos_id())

assert target_ids[-1] == sp.eos_id(), (
    "STOP: EOS is NOT present in the supervised target. "
    "Do not train until this is fixed."
)

print("\nSFT dataset + EOS supervision: PASS")

INPUT:
કુશલ દ્વારા કાપવામાં આવેલું અંતર કિરણ કરતાં ઓછું છે.
કિરણ અને કુશલમાંથી કોણે વધુ અંતર કાપ્યું છે?

GOLD ANSWER:
કિરણ

TARGET IDS:
[7831, 2]

TARGET DECODED:
કિરણ

EOS ID: 2
LAST TARGET ID: 2
LAST TARGET IS EOS?: True

SFT dataset + EOS supervision: PASS


In [116]:
# ============================================================
# COLLATE
# ============================================================

def collate_sft(batch):
    max_len = max(len(x["input_ids"]) for x in batch)

    pad_id = sp.pad_id()
    if pad_id < 0:
        pad_id = 0

    input_batch = []
    label_batch = []

    for item in batch:
        input_ids = item["input_ids"]
        labels = item["labels"]

        pad_len = max_len - len(input_ids)

        input_ids = torch.cat([
            input_ids,
            torch.full(
                (pad_len,),
                pad_id,
                dtype=torch.long
            )
        ])

        labels = torch.cat([
            labels,
            torch.full(
                (pad_len,),
                -100,
                dtype=torch.long
            )
        ])

        input_batch.append(input_ids)
        label_batch.append(labels)

    return {
        "input_ids": torch.stack(input_batch),
        "labels": torch.stack(label_batch),
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_sft,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_sft,
    pin_memory=torch.cuda.is_available(),
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Train batches: 1600
Val batches: 200


In [117]:
# ============================================================
# BATCH SANITY CHECK
# ============================================================

batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Label shape:", batch["labels"].shape)

target = batch["labels"][0]
target_ids = target[target != -100].tolist()

print("Decoded supervised target:")
print(sp.decode(target_ids))

assert target_ids[-1] == sp.eos_id()

print("Batch EOS check: PASS")

Input shape: torch.Size([4, 56])
Label shape: torch.Size([4, 56])
Decoded supervised target:
મનોજ
Batch EOS check: PASS


In [118]:
# ============================================================
# MODEL OUTPUT API SANITY CHECK
# ============================================================

with torch.no_grad():
    outputs = model(batch["input_ids"].to(device))

print("Model output type:", type(outputs))
print("Logits shape:", outputs[0].shape)
print("Labels shape:", batch["labels"].shape)

assert outputs[0].shape[0] == batch["input_ids"].shape[0]
assert outputs[0].shape[1] == batch["input_ids"].shape[1]
assert outputs[0].shape[2] == config.vocab_size

print("Model output API check: PASS")


Model output type: <class 'tuple'>
Logits shape: torch.Size([4, 56, 10000])
Labels shape: torch.Size([4, 56])
Model output API check: PASS


In [119]:
# ============================================================
# SFT HYPERPARAMETERS
# ============================================================

EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUMULATION = 8

LR = 2e-4
WEIGHT_DECAY = 0.01
BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0

AMP_ENABLED = torch.cuda.is_available()

OUTPUT_DIR = Path("/kaggle/working/gujarati_phase3_sft")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /kaggle/working/gujarati_phase3_sft


In [120]:
# ============================================================
# FRESH PHASE-3 OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)

print("Fresh Phase-3 AdamW optimizer created.")


Fresh Phase-3 AdamW optimizer created.


In [121]:
# ============================================================
# TRAIN / VALIDATE
# ============================================================

import torch.nn.functional as F

def get_logits(model, input_ids):
    """Call the existing Phase-2 model API and extract logits."""
    outputs = model(input_ids)
    assert isinstance(outputs, (tuple, list)), (
        f"Unexpected model output type: {type(outputs)}"
    )
    logits = outputs[0]
    return logits


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.amp.autocast(
            "cuda",
            enabled=AMP_ENABLED
        ):
            logits = get_logits(model, input_ids)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100
            )

        scaler.scale(loss / GRAD_ACCUMULATION).backward()

        if (
            (batch_idx + 1) % GRAD_ACCUMULATION == 0
            or (batch_idx + 1) == len(loader)
        ):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), GRAD_CLIP
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.amp.autocast(
            "cuda",
            enabled=AMP_ENABLED
        ):
            logits = get_logits(model, input_ids)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100
            )

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches

print("Training functions loaded: no labels= argument is used.")


Training functions loaded: no labels= argument is used.


In [122]:
# ============================================================
# CHECKPOINT SAVING
# ============================================================

def save_checkpoint(
    path,
    epoch,
    train_loss,
    val_loss
):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "phase": 3,
        "stage": "SFT",
        "vocab_size": sp.vocab_size(),
        "eos_id": sp.eos_id(),
    }

    torch.save(payload, path)
    print("Saved:", path)

In [123]:
# ============================================================
# SFT TRAINING
# ============================================================

best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):

    start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler
    )

    val_loss = evaluate_loss(
        model,
        val_loader
    )

    elapsed = (time.time() - start) / 60
    val_ppl = math.exp(min(val_loss, 20))

    print()
    print("=" * 70)
    print(f"EPOCH {epoch}/{EPOCHS}")
    print("=" * 70)
    print(f"Train CE : {train_loss:.4f}")
    print(f"Val CE   : {val_loss:.4f}")
    print(f"Val PPL  : {val_ppl:.4f}")
    print(f"Time     : {elapsed:.2f} min")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_ppl": val_ppl,
    })

    epoch_path = (
        CHECKPOINT_DIR
        / f"phase3_gujarati_sft_epoch_{epoch}.pt"
    )

    save_checkpoint(
        epoch_path,
        epoch,
        train_loss,
        val_loss
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_path = (
            CHECKPOINT_DIR
            / "phase3_gujarati_sft_best.pt"
        )

        save_checkpoint(
            best_path,
            epoch,
            train_loss,
            val_loss
        )

        print("NEW BEST MODEL")


EPOCH 1/5
Train CE : 0.7259
Val CE   : 0.2893
Val PPL  : 1.3355
Time     : 0.68 min
Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_1.pt
Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_best.pt
NEW BEST MODEL

EPOCH 2/5
Train CE : 0.2301
Val CE   : 0.1675
Val PPL  : 1.1824
Time     : 0.68 min
Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_2.pt
Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_best.pt
NEW BEST MODEL

EPOCH 3/5
Train CE : 0.1056
Val CE   : 0.0375
Val PPL  : 1.0383
Time     : 0.68 min
Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_3.pt
Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_best.pt
NEW BEST MODEL

EPOCH 4/5
Train CE : 0.0579
Val CE   : 0.0233
Val PPL  : 1.0236
Time     : 0.69 min
Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_4.pt
Saved: /kaggle/working

In [124]:
# ============================================================
# FINAL CHECKPOINT
# ============================================================

final_path = (
    CHECKPOINT_DIR
    / "phase3_gujarati_sft_final.pt"
)

save_checkpoint(
    final_path,
    EPOCHS,
    history[-1]["train_loss"],
    history[-1]["val_loss"]
)

print("\nFinal checkpoint:", final_path)

Saved: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_final.pt

Final checkpoint: /kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_final.pt


In [125]:
# ============================================================
# GENERATION
# ============================================================

@torch.no_grad()
def generate_answer(prompt, max_new_tokens=32):
    model.eval()

    prompt_text = prompt + "\nજવાબ: "
    prompt_ids = sp.encode(prompt_text, out_type=int)

    input_ids = torch.tensor(
        [prompt_ids], dtype=torch.long, device=device
    )

    generated = []

    for _ in range(max_new_tokens):
        if input_ids.size(1) > SEQ_LEN:
            input_ids = input_ids[:, -SEQ_LEN:]

        outputs = model(input_ids)
        logits = outputs[0]

        next_token = torch.argmax(
            logits[:, -1, :], dim=-1
        )

        token_id = next_token.item()

        # Stop only when actual EOS is generated.
        if token_id == sp.eos_id():
            break

        generated.append(token_id)
        input_ids = torch.cat(
            [input_ids, next_token.view(1, 1)], dim=1
        )

    return sp.decode(generated).strip()


## Pre-evaluation generation checks

These checks are deliberately before the full 800-example evaluation.

1. Normal single-answer task.
2. Ordering task, which must generate **all entities**, not just the first one.

In [126]:
# ============================================================
# TRAINING EXAMPLES
# ============================================================

print("=" * 70)
print("TRAINING EXAMPLE GENERATION")
print("=" * 70)

for i in range(min(10, len(train_rows))):

    row = train_rows[i]

    pred = generate_answer(
        row["input"],
        max_new_tokens=32
    )

    print()
    print(f"Example {i + 1}")
    print("Task:", row.get("task"))
    print("Gold:", row["answer"])
    print("Pred:", pred)

TRAINING EXAMPLE GENERATION

Example 1
Task: reversed_wording
Gold: કિરણ
Pred: કિરણ

Example 2
Task: three_hop
Gold: પંકજ
Pred: પંકજ

Example 3
Task: ordering
Gold: મનોજ, કિરણ, ગૌરવ, સમીર
Pred: મનોજ, ગૌરવ, સમીર

Example 4
Task: equal
Gold: સમાન
Pred: સમાન

Example 5
Task: greater_smaller
Gold: નિખિલ
Pred: નિખિલ

Example 6
Task: ordering
Gold: ગૌરવ, હિતેશ, સંદીપ, વરુણ
Pred: ગૌરવ, હિતેશ, સંદીપ, વરુણ

Example 7
Task: equal
Gold: સમાન
Pred: સમાન

Example 8
Task: greater_smaller
Gold: મનોજ
Pred: મનોજ

Example 9
Task: equal
Gold: સમાન
Pred: સમાન

Example 10
Task: two_hop
Gold: ધ્રુવ
Pred: ધ્રુવ


In [127]:
# ============================================================
# ORDERING RAW GENERATION CHECK
# ============================================================

ordering_example = next(
    row for row in test_rows
    if row.get("task") == "ordering"
)

ordering_pred = generate_answer(
    ordering_example["input"],
    max_new_tokens=32
)

print("INPUT:")
print(ordering_example["input"])

print("\nGOLD:")
print(ordering_example["answer"])

print("\nRAW PREDICTION:")
print(ordering_pred)

print("\nIMPORTANT: For ordering, the complete list is retained.")

INPUT:
ગૌરવની ઊંચાઈ સંદીપ કરતાં વધારે છે.
નિખિલની ઊંચાઈ ગૌરવ કરતાં વધુ છે.
વિવેક નિખિલ કરતાં ઊંચો છે.
ઊંચાઈ પ્રમાણે નાનાથી મોટાના ક્રમમાં ગોઠવો.

GOLD:
સંદીપ, ગૌરવ, નિખિલ, વિવેક

RAW PREDICTION:
સંદીપ, ગૌરવ, નિખિલ, વિવેક

IMPORTANT: For ordering, the complete list is retained.


In [128]:
# ============================================================
# TASK-AWARE NORMALIZATION
# ============================================================

def normalize_prediction(text, task):

    text = text.strip()

    if "જવાબ:" in text:
        text = text.split("જવાબ:")[-1].strip()

    # Keep first generated line only.
    text = text.split("\n")[0].strip()

    if task == "ordering":
        # DO NOT truncate ordering to first entity.
        # Preserve the entire comma-separated list.
        parts = [
            x.strip()
            for x in text.split(",")
            if x.strip()
        ]

        return ", ".join(parts)

    # All other tasks have a single answer entity.
    return text.split(",")[0].strip()

In [129]:
# ============================================================
# TEST EVALUATION
# ============================================================

def evaluate_test_set(rows):

    predictions = []
    correct = 0

    for i, row in enumerate(rows):

        pred_raw = generate_answer(
            row["input"],
            max_new_tokens=32
        )

        task = row.get("task")

        pred = normalize_prediction(
            pred_raw,
            task
        )

        gold = normalize_prediction(
            row["answer"],
            task
        )

        is_correct = pred == gold

        if is_correct:
            correct += 1

        predictions.append({
            "id": row.get("id"),
            "input": row["input"],
            "gold": gold,
            "prediction_raw": pred_raw,
            "prediction": pred,
            "correct": is_correct,
            "task": task,
            "difficulty": row.get("difficulty"),
            "domain": row.get("domain"),
        })

        if (i + 1) % 100 == 0:
            print(f"{i + 1}/{len(rows)}")

    accuracy = correct / len(rows)

    return predictions, correct, accuracy


predictions, correct, accuracy = evaluate_test_set(
    test_rows
)

print()
print("=" * 70)
print("GUJARATI PHASE-3 TEST")
print("=" * 70)
print(f"Correct : {correct}")
print(f"Total   : {len(test_rows)}")
print(f"Accuracy: {accuracy * 100:.2f}%")

100/800
200/800
300/800
400/800
500/800
600/800
700/800
800/800

GUJARATI PHASE-3 TEST
Correct : 785
Total   : 800
Accuracy: 98.12%


In [130]:
# ============================================================
# GROUPED ACCURACY
# ============================================================

def grouped_accuracy(predictions, key):

    groups = defaultdict(lambda: [0, 0])

    for row in predictions:
        value = row[key]

        groups[value][1] += 1

        if row["correct"]:
            groups[value][0] += 1

    return groups


task_groups = grouped_accuracy(
    predictions,
    "task"
)

print("\nTASK")
print("-" * 50)

for task, (c, n) in sorted(task_groups.items()):
    print(
        f"{task:22s} "
        f"{c:3d}/{n:3d} "
        f"{100*c/n:6.2f}%"
    )


difficulty_groups = grouped_accuracy(
    predictions,
    "difficulty"
)

print("\nDIFFICULTY")
print("-" * 50)

for difficulty, (c, n) in sorted(difficulty_groups.items()):
    print(
        f"{str(difficulty):22s} "
        f"{c:3d}/{n:3d} "
        f"{100*c/n:6.2f}%"
    )


domain_groups = grouped_accuracy(
    predictions,
    "domain"
)

print("\nDOMAIN")
print("-" * 50)

for domain, (c, n) in sorted(domain_groups.items()):
    print(
        f"{domain:22s} "
        f"{c:3d}/{n:3d} "
        f"{100*c/n:6.2f}%"
    )


TASK
--------------------------------------------------
direct_comparison      101/101 100.00%
distractor             101/101 100.00%
equal                   94/ 94 100.00%
greater_smaller        100/100 100.00%
ordering                86/101  85.15%
reversed_wording       101/101 100.00%
three_hop              101/101 100.00%
two_hop                101/101 100.00%

DIFFICULTY
--------------------------------------------------
1                      295/295 100.00%
2                      202/202 100.00%
3                      288/303  95.05%

DOMAIN
--------------------------------------------------
age                    118/121  97.52%
distance               118/122  96.72%
height                 143/146  97.95%
score                  128/130  98.46%
speed                  122/125  97.60%
weight                 156/156 100.00%


In [131]:
# ============================================================
# SAVE PREDICTIONS + METRICS
# ============================================================

predictions_path = (
    OUTPUT_DIR
    / "gujarati_phase3_test_predictions.jsonl"
)

with open(
    predictions_path,
    "w",
    encoding="utf-8"
) as f:
    for row in predictions:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False
            ) + "\n"
        )

metrics = {
    "language": "Gujarati",
    "phase": 3,
    "stage": "SFT",

    "train_size": len(train_rows),
    "val_size": len(val_rows),
    "test_size": len(test_rows),

    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accumulation": GRAD_ACCUMULATION,

    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,

    "best_val_loss": best_val_loss,
    "final_train_loss": history[-1]["train_loss"],
    "final_val_loss": history[-1]["val_loss"],

    "test_correct": correct,
    "test_total": len(test_rows),
    "test_accuracy": accuracy,

    "vocab_size": sp.vocab_size(),
    "eos_id": sp.eos_id(),

    "history": history,
}

metrics_path = (
    OUTPUT_DIR
    / "gujarati_phase3_metrics.json"
)

with open(
    metrics_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metrics,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Predictions:", predictions_path)
print("Metrics    :", metrics_path)

Predictions: /kaggle/working/gujarati_phase3_sft/gujarati_phase3_test_predictions.jsonl
Metrics    : /kaggle/working/gujarati_phase3_sft/gujarati_phase3_metrics.json


In [132]:
# ============================================================
# FINAL OUTPUT SUMMARY
# ============================================================

print("=" * 70)
print("PHASE 3 OUTPUTS")
print("=" * 70)

for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(p)

print()
print("Final checkpoint:")
print(final_path)

print()
print("Best checkpoint:")
print(CHECKPOINT_DIR / "phase3_gujarati_sft_best.pt")

PHASE 3 OUTPUTS
/kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_best.pt
/kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_1.pt
/kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_2.pt
/kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_3.pt
/kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_4.pt
/kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_epoch_5.pt
/kaggle/working/gujarati_phase3_sft/checkpoints/phase3_gujarati_sft_final.pt
/kaggle/working/gujarati_phase3_sft/gujarati_phase3_metrics.json
/kaggle/working/gujarati_phase3_sft/gujarati_phase3_test_predictions.jsonl
/kaggle/working/gujarati_phase3_sft/logs/sft_training_log.csv
/kaggle/working/gujarati_phase3_sft/predictions/gujarati_phase3_metrics.json
/kaggle/working/gujarati_phase3_sft/predictions/gujarati_phase3_test_predictions.jsonl

Final checkpoint:
/kaggle/working/gujarati_phase3_sft/checkpoints/pha

In [133]:
# ============================================
# ADD BEST SFT CHECKPOINT TO KAGGLE DATASET
# ============================================

from pathlib import Path
import shutil
import json
import os

BEST_CKPT = Path(
    "/kaggle/working/gujarati_phase3_sft/checkpoints/"
    "phase3_gujarati_sft_best.pt"
)

UPLOAD_DIR = Path("/kaggle/working/guj_ckpt_upload")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

assert BEST_CKPT.exists(), f"Checkpoint not found: {BEST_CKPT}"

# Copy best checkpoint
shutil.copy2(
    BEST_CKPT,
    UPLOAD_DIR / "phase3_gujarati_sft_best.pt"
)

# Copy metrics if available
METRICS = Path(
    "/kaggle/working/gujarati_phase3_sft/"
    "gujarati_phase3_metrics.json"
)

if METRICS.exists():
    shutil.copy2(
        METRICS,
        UPLOAD_DIR / "gujarati_phase3_metrics.json"
    )

# --------------------------------------------
# REQUIRED KAGGLE METADATA
# --------------------------------------------

metadata = {
    "title": "Gujarati LMA Checkpoints and JSON",
    "id": "vidurshah04/guj-ckcpts-and-json",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ]
}

with open(UPLOAD_DIR / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Upload folder:")
for f in UPLOAD_DIR.iterdir():
    print(" ", f.name)

# --------------------------------------------
# CREATE NEW DATASET VERSION
# --------------------------------------------

!kaggle datasets version \
    -p /kaggle/working/guj_ckpt_upload \
    -m "Add best Gujarati Phase 3 SFT checkpoint new LR"

Upload folder:
  phase3_gujarati_sft_best.pt
  gujarati_phase3_metrics.json
  dataset-metadata.json
Starting upload for file phase3_gujarati_sft_best.pt
100%|████████████████████████████████████████| 295M/295M [00:08<00:00, 38.3MB/s]
Upload successful: phase3_gujarati_sft_best.pt (295MB)
Starting upload for file gujarati_phase3_metrics.json
100%|██████████████████████████████████████| 1.19k/1.19k [00:00<00:00, 3.09kB/s]
Upload successful: gujarati_phase3_metrics.json (1KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/vidurshah04/guj-ckcpts-and-json
